# Tile Stitching / Mapping

This notebook stitches image tiles into a single large map using their
geospatial bounding boxes (RD coordinates) from a metadata JSON file.

It is used to build full-area maps from four different sources:

1. **GT map**     – high-resolution ground-truth tiles
2. **LR map**     – low-resolution input tiles
3. **Satlas map** – super-resolution output tiles from Satlas
4. **SwinIR map** – super-resolution output tiles from SwinIR

> Note: this notebook performs **no inference** — it only assembles
> already-produced tiles into a single mosaic. Model inference and
> evaluation metrics live in their own notebooks.

## Example metadata format

```json
{
  "tiles": [
    {
      "tile_id": "tile_r0017_c0015",
      "bbox_rd": [138880.0, 460800.0, 139520.0, 461440.0]
    }
  ]
}
```

Tile filenames (without extension and without any model suffix) must
match `tile_id` in the metadata.

In [ ]:
# Imports
from pathlib import Path
import json
import cv2
import numpy as np

In [ ]:
# Configuration
# Adjust these paths and per-model settings to your local setup.

# Path to metadata JSON containing tile_id -> bbox_rd mapping
METADATA_PATH = Path('data/tiles_metadata.json')

# Tile directories
GT_TILES_DIR     = Path('data/GT')
LR_TILES_DIR     = Path('data/inputs')
SATLAS_TILES_DIR = Path('data/satlas_results')
SWINIR_TILES_DIR = Path('data/swinir_results')

# Per-model file extensions and filename suffixes.
# Suffix is stripped from the filename stem before matching against tile_id.
# Inspect a few filenames in each directory to confirm these are correct.
GT_GLOB,     GT_SUFFIX     = '*.jpg', ''
LR_GLOB,     LR_SUFFIX     = '*.jpg', ''
SATLAS_GLOB, SATLAS_SUFFIX = '*.png', ''         # Satlas typically writes <tile_id>.png
SWINIR_GLOB, SWINIR_SUFFIX = '*.png', '_SwinIR'  # SwinIR writes <tile_id>_SwinIR.png

# Output paths for stitched maps
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GT_OUTPUT     = OUTPUT_DIR / 'GT_fullmap.png'
LR_OUTPUT     = OUTPUT_DIR / 'LR_fullmap.png'
SATLAS_OUTPUT = OUTPUT_DIR / 'Satlas_fullmap.png'
SWINIR_OUTPUT = OUTPUT_DIR / 'SwinIR_fullmap.png'

---

## Stitching function

Tiles are placed on a canvas according to their bounding-box position in
the RD coordinate system. Each unique `x0` becomes a column index, each
unique `y0` becomes a row index. The Y axis is inverted so that higher
Y (north) appears at the top of the image.

An optional `filename_suffix` lets you strip suffixes that the model
appended to its outputs (e.g. SwinIR appends `_SwinIR` to tile names).

In [ ]:
def load_metadata(json_path: Path) -> dict:
    """Load tile metadata and return {tile_id: tile_info}."""
    with open(json_path, 'r') as f:
        data = json.load(f)

    return {
        str(t['tile_id']).lower().strip(): t
        for t in data['tiles']}


def stitch_tiles(
    tiles_dir: Path,
    metadata_lookup: dict,
    output_path: Path,
    file_glob: str = '*.png',
    filename_suffix: str = '',
) -> np.ndarray:
    """Stitch all tiles in `tiles_dir` into one image using bbox metadata.

    Parameters
    ----------
    tiles_dir : Path
        Directory containing tile images.
    metadata_lookup : dict
        Mapping {tile_id: {'bbox_rd': [x0, y0, x1, y1], ...}}.
    output_path : Path
        Where to save the stitched map.
    file_glob : str
        Glob pattern for tile files (e.g. '*.jpg', '*.png').
    filename_suffix : str
        Suffix appended by the model to filenames; stripped before lookup.
        Example: 'tile001_SwinIR.png' with suffix '_SwinIR' -> 'tile001'.
    """
    suffix = filename_suffix.lower()

    tile_map: dict = {}
    xs, ys = [], []
    skipped = 0

    for p in tiles_dir.glob(file_glob):
        stem = p.stem.lower().strip()
        tile_id = stem[:-len(suffix)] if suffix and stem.endswith(suffix) else stem

        meta = metadata_lookup.get(tile_id)
        if meta is None:
            skipped += 1
            continue

        x0, y0, x1, y1 = meta['bbox_rd']
        tile_map[tile_id] = {'path': p, 'bbox': (x0, y0, x1, y1)}
        xs.append(x0)
        ys.append(y0)

    if not tile_map:
        raise RuntimeError(
            f'No matching tiles found in {tiles_dir} (glob={file_glob}, '
            f'suffix={filename_suffix!r}). Check filename convention.')

    print(f'Matched {len(tile_map)} tiles from {tiles_dir} '
          f'(skipped {skipped} without metadata)')

    # Build grid index from unique coordinates (Y inverted: north on top)
    xs_unique = sorted(set(xs))
    ys_unique = sorted(set(ys), reverse=True)
    x_index = {x: i for i, x in enumerate(xs_unique)}
    y_index = {y: i for i, y in enumerate(ys_unique)}

    # Determine tile size from a sample
    sample_path = next(iter(tile_map.values()))['path']
    sample = cv2.imread(str(sample_path), cv2.IMREAD_COLOR)
    if sample is None:
        raise RuntimeError(f'Could not read sample tile: {sample_path}')
    h, w = sample.shape[:2]

    canvas = np.zeros(
        (len(ys_unique) * h, len(xs_unique) * w, 3),
        dtype=np.uint8)
    
    print(f'Canvas size: {canvas.shape[1]} x {canvas.shape[0]} '
          f'(tile {w}x{h}, grid {len(xs_unique)}x{len(ys_unique)})')

    for tile_id, info in tile_map.items():
        img = cv2.imread(str(info['path']), cv2.IMREAD_COLOR)
        if img is None:
            print(f'Could not load: {tile_id}')
            continue
        if img.shape[:2] != (h, w):
            print(f'Size mismatch for {tile_id}: {img.shape[:2]} vs {(h, w)}')
            continue

        x0, y0, _, _ = info['bbox']
        x = x_index[x0] * w
        y = y_index[y0] * h
        canvas[y:y + h, x:x + w] = img

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), canvas)
    print(f'Saved: {output_path}')

    return canvas

## Load metadata

In [ ]:
metadata_lookup = load_metadata(METADATA_PATH)
print(f'Loaded metadata for {len(metadata_lookup)} tiles')

## 1. GT map

In [ ]:
_ = stitch_tiles(
    tiles_dir=GT_TILES_DIR,
    metadata_lookup=metadata_lookup,
    output_path=GT_OUTPUT,
    file_glob=GT_GLOB,
    filename_suffix=GT_SUFFIX)

## 2. LR map

In [ ]:
_ = stitch_tiles(
    tiles_dir=LR_TILES_DIR,
    metadata_lookup=metadata_lookup,
    output_path=LR_OUTPUT,
    file_glob=LR_GLOB,
    filename_suffix=LR_SUFFIX)

## 3. Satlas map

If this cell raises `No matching tiles found`, inspect a few filenames in
`SATLAS_TILES_DIR` and adjust `SATLAS_GLOB` and `SATLAS_SUFFIX` in the
config cell accordingly.

In [ ]:
_ = stitch_tiles(
    tiles_dir=SATLAS_TILES_DIR,
    metadata_lookup=metadata_lookup,
    output_path=SATLAS_OUTPUT,
    file_glob=SATLAS_GLOB,
    filename_suffix=SATLAS_SUFFIX)

## 4. SwinIR map

In [ ]:
_ = stitch_tiles(
    tiles_dir=SWINIR_TILES_DIR,
    metadata_lookup=metadata_lookup,
    output_path=SWINIR_OUTPUT,
    file_glob=SWINIR_GLOB,
    filename_suffix=SWINIR_SUFFIX)